# Validasi temporal, kelayakan aktual, dan paket rilis model

**DATA AKTUAL BELUM TERSEDIA;** dry-run protokol memakai **DATA SINTETIS / SIMULASI**. Notebook memeriksa kesiapan data aktual melalui gate kelayakan (data_readiness), mendemonstrasikan penolakan data tidak layak, serta memverifikasi protokol split temporal dengan purge target, preprocessing fit-only, dan pembuatan paket rilis terpisah.

Target tetap future_7d_mean_sbp. Artefak lama CP3, P2, dan Minggu 10 tidak diubah. Hasil tersimpan di models/releases/synthetic-protocol-v1/ dan reports/actual_model/.

In [1]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if not (ROOT / "backend").is_dir(): ROOT = ROOT.parent
assert (ROOT / "backend/app/actual_experiment.py").is_file()
sys.path.insert(0, str(ROOT))

sys.path.insert(0, str(ROOT / "backend"))
from etl.train_actual_model import execute, legacy_artifact_hashes
before_hashes = legacy_artifact_hashes()
summary = execute(run_synthetic_protocol=True)
print(f"Aktual eligible={summary['actual_status']['eligible']} ({summary['actual_status']['reasons']})")
dry = summary["protocol_dry_run"]
print(f"Dry-run {dry['release_id']} | run {dry['run_id']} | terpilih {dry['selected_model']} | beats_naive={dry['beats_naive_on_validation']}")

ModuleNotFoundError: No module named 'backend.app.data_readiness'

Aktual eligible=False (['actual_export_files_missing'])
Dry-run synthetic-protocol-v1 | run b00d9f2377eb2b26c33e | terpilih Linear Regression | beats_naive=False

In [ ]:
import pandas as pd
report = ROOT / "reports/actual_model/protocol_dry_run"
comp_df = pd.read_csv(report / "comparison_summary.csv")
pred_df = pd.read_csv(report / "test_predictions.csv")
display(comp_df)
display(pred_df)

metrics_doc = json.loads((report / "experiment_metrics.json").read_text(encoding="utf-8"))
print("Strategi CV:", metrics_doc["selection"]["cv_strategy"])
print("Audit leakage:", metrics_doc["leakage_audit"])

In [ ]:
from app.actual_experiment import assess_experiment_eligibility, verify_release_package
dates = pd.date_range("2026-10-01", periods=16, freq="D").strftime("%Y-%m-%d")
short_prof = pd.DataFrame([{"respondent_id": f"AK{i}", "age": 30 + i, "bmi": 23.0} for i in range(3)])
short_day = pd.DataFrame([{"respondent_id": f"AK{i}", "date": d, "sleep_hours": 7.0, "stress": 2, "activity_minutes": 30} for i in range(3) for d in dates])
short_ses = pd.DataFrame([{"respondent_id": f"AK{i}", "date": d, "session": "Pagi", "sbp1": 120, "dbp1": 80, "sbp2": 121, "dbp2": 81} for i in range(3) for d in dates])
rejected = assess_experiment_eligibility(short_prof, short_ses, short_day, {}, source_kind="actual")
print("Simulasi data aktual pendek -> eligible:", rejected["eligible"], "| alasan:", rejected["reasons"])
model, manifest, _ = verify_release_package(ROOT / "models/releases/synthetic-protocol-v1")
after_hashes = legacy_artifact_hashes()
assert before_hashes == after_hashes
print(f"Paket rilis terverifikasi ({manifest['release_id']}); {len(after_hashes)} artefak lama CP3/P2/Minggu 10 utuh.")

In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
x = range(len(comp_df))
w = 0.36
axes[0].bar([i - w/2 for i in x], comp_df["val_rmse"], width=w, label="RMSE Validasi")
axes[0].bar([i + w/2 for i in x], comp_df["test_rmse"], width=w, label="RMSE Test")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(comp_df["model"], rotation=20, ha="right", fontsize=8)
axes[0].set_ylabel("RMSE (mmHg)")
axes[0].set_title("Perbandingan RMSE Validasi vs Test (Sintetis)")
axes[0].legend()
axes[1].scatter(pred_df["actual"], pred_df["linear_regression"], label="Linear Regression", alpha=0.85)
axes[1].scatter(pred_df["actual"], pred_df["naive_7_day_mean"], label="Baseline Naif", marker="x")
lo = min(pred_df["actual"].min(), pred_df["linear_regression"].min()) - 1
hi = max(pred_df["actual"].max(), pred_df["linear_regression"].max()) + 1
axes[1].plot([lo, hi], [lo, hi], "--", color="gray", label="Aktual = Prediksi")
axes[1].set_xlabel("Aktual (mmHg)")
axes[1].set_ylabel("Prediksi (mmHg)")
axes[1].set_title("Aktual vs Prediksi (Test Set, n=7)")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

NameError: name 'comp_df' is not defined

## Interpretasi

Gate kelayakan menolak eksperimen aktual karena berkas data/actual/ belum tersedia dan menolak sampel pendek secara otomatis. Pada dry-run fixture sintetis, kalender training terlalu pendek untuk inner CV kronologis murni sehingga dicatat jujur sebagai respondent_group_kfold_non_chronological_short_calendar. Baseline naif tetap mengungguli seluruh model terlatih pada data sintetis ini. Paket rilis models/releases/synthetic-protocol-v1/ berstatus candidate (source_kind=synthetic, target_approval=false) dan tidak menggantikan model serving aktif sebelum data aktual layak serta disepakati bersama Hugo dan dosen.